In [149]:
# Standard library
import os
import re
import json
import subprocess
import shutil
from pathlib import Path
from pprint import pprint

# Third-party libraries
import numpy as np
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import faiss
import torch
from elasticsearch import Elasticsearch, helpers
import matplotlib.pyplot as plt
import seaborn as sns

# Jupyter / display utilities
from IPython.display import Markdown, display


In [150]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [151]:
def preprocess(s):
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"http\S+|www\.\S+", "<URL>", s)
    s = re.sub(r"\S+@\S+", "<EMAIL>", s)
    s = re.sub(r"<[^>]+>", " ", s)
    s = ''.join(ch for ch in s if ord(ch) >= 32)
    s = re.sub(r"['\"]", "", s)
    return s.strip()

In [152]:
from pathlib import Path

# Base project folders
BASE_DIR = Path.cwd()
IR_DIR   = BASE_DIR / "IR2025"
EMBED_DIR = BASE_DIR / "Embeddings"

IR_DIR.mkdir(exist_ok=True)
EMBED_DIR.mkdir(exist_ok=True)

# Core dataset paths
DOCS_CSV        = IR_DIR / "documents.csv"
DOCS_JSONL      = IR_DIR / "documents.jsonl"
QUERIES_CSV     = IR_DIR / "queries.csv"

# Embeddings
DOC_EMBED_FILE   = EMBED_DIR / "doc_embeddings3.npy"
QUERY_EMBED_FILE = EMBED_DIR / "query_embeddings3.npy"

# Relevance files (CSV → TXT for trec_eval)
QRELS_CSV = IR_DIR / "qrels.csv"
QRELS_TXT = IR_DIR / "qrels.txt"

# trec_eval binary
TREC_EVAL_BIN = IR_DIR / "trec_eval" / "trec_eval.exe"


In [153]:
# hyperparams
DOC_BATCH = 64
QUERY_BATCH = 32
FAISS_USE_COSINE = True
PRF_ENABLED = False              # αν θέλεις PRF ενεργό
PRF_TOP_M = 5                   # top-m docs for feedback
PRF_ALPHA = 0.7                 # weight for original query
PRF_BETA = 0.3                  # weight for feedback mean
REQUERY_CANDIDATES = 200        # initial candidate size if using candidate-limited PRF (not required)
RESULT_KS = (20, 30, 50)


In [154]:
if not DOCS_CSV.exists():
    raise SystemExit(f"documents.csv not found: {DOCS_CSV}")
if not QUERIES_CSV.exists():
    raise SystemExit(f"queries.csv not found: {QUERIES_CSV}")

df_docs = pd.read_csv(DOCS_CSV)
df_queries = pd.read_csv(QUERIES_CSV)

In [155]:
# Apply preprocess and ensure ID are strings
df_docs = df_docs.dropna(subset=["Text"]).copy()
df_docs["Text"] = df_docs["Text"].astype(str).map(preprocess)
df_docs["ID"] = df_docs["ID"].astype(str).str.strip()

df_queries = df_queries.dropna(subset=["Text"]).copy()
df_queries["Text"] = df_queries["Text"].astype(str).map(preprocess)
df_queries["ID"] = df_queries["ID"].astype(str).str.strip()

print(f"Loaded {len(df_docs)} documents; {len(df_queries)} queries.")



Loaded 18316 documents; 10 queries.


In [156]:
model = SentenceTransformer("all-mpnet-base-v2",device=device)

In [157]:
# Convert to JSONL
records_written = 0
df_docs["ID"] = df_docs["ID"].astype(str).str.strip()
with open(DOCS_JSONL, "w", encoding="utf-8") as f:
    for _, row in df_docs.iterrows():
        record = {
            "id": str(row["ID"]).strip(),
            "text": row["Text"]
        }
        json_line = json.dumps(record, ensure_ascii=False)
        f.write(json_line + "\n")
        records_written += 1

print(f"Converted {records_written} rows → JSONL format")
print(f"Output saved at: {DOCS_JSONL.resolve()}")

Converted 18316 rows → JSONL format
Output saved at: C:\Users\perik\Downloads\Semantic-Information-Retrieval-System-ElasticSearch-FAISS-Transformers-\IR2025\documents.jsonl


In [158]:
class Search:
    def __init__(self):
        self.es = Elasticsearch("http://127.0.0.1:9200")
        client_info = self.es.info()
        print('Connected to Elasticsearch!')
        pprint(client_info.body)

    #Return instance
    def get_es(self):
        return self.es

    #Creates empty index with the parameters we feed
    def create_index(self):
        index_settings = {
            "settings": {
                "similarity": {
                    "default": {"type": "BM25", "k1": 1.4, "b": 0.7}
                },
                "analysis": {
                    "analyzer": {
                        "my_english_analyzer": {
                            "type": "custom",
                            "tokenizer": "standard",
                            "filter": [
                                "lowercase",
                                "english_stop",
                                "porter_stem"
                            ]
                        }
                    },
                    "filter": {
                        "english_stop": {
                            "type": "stop",
                            "stopwords": "_english_"
                        }
                    }
                }
            },
            "mappings": {
                "properties": {
                    "id": {"type": "keyword"},
                    "text": {"type": "text", "analyzer": "my_english_analyzer"}
                }
            }
        }

        self.es.indices.create(index='ir2025', body=index_settings)
        print("Index created with English analyzer and BM25 similarity.")

    #Insert single document
    def insert_document(self, document):
        return self.es.index(index='ir2025', document=document)

    #Insert multiple indexes
    def insert_documents(self, documents):
        operations = []
        for document in documents:
            operations.append({'index': {'_index': 'ir2025'}})
            operations.append(document)

        result = self.es.bulk(operations=operations)
        print("Insertion finished.")
        print(result)

    #Deletes index
    def delete_index(self, index_name="ir2025"):
        if self.es.indices.exists(index=index_name):
            self.es.indices.delete(index=index_name)
            print(f"Index '{index_name}' deleted successfully.")
        else:
            print(f"Index '{index_name}' does not exist.")

    #Check if index exists
    def exists(self, index_name="ir2025"):
        try:
            return self.es.indices.exists(index=index_name)
        except Exception as e:
            print(f"Error checking if index exists: {e}")
            return False

    #Helper function for bulk indexing
    def generate_actions(self, jsonl_path, index_name):
        with open(jsonl_path, encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue 
                doc = json.loads(line)
                doc_id = str(doc["id"]).strip()
                # Ensure cleaned ID is saved back to _source
                doc["id"] = doc_id
                yield {
                    "_index": index_name,
                    "_id": doc_id,     
                    "_source": doc
                }

    #Bulk index docs
    def index_documents(self, jsonl_path, index_name):
        actions = self.generate_actions(jsonl_path, index_name)
        success, _ = helpers.bulk(self.get_es(), actions)
        print(f"Successfully indexed {success} documents into '{index_name}'")

    #Runs query
    def search_query(self, query_text, k=10, index_name="ir2025"):
        try:
            resp = self.es.search(
                index=index_name,
                query={"match": {"text": {"query": query_text}}},
                size=k
            )
            return resp["hits"]["hits"]
        except Exception as e:
            print(f"Error executing search query: {e}")
            return []



In [159]:
class Evaluation_BM25:
    #Return relative paths
    def _rel(self, p: Path):
        try:
            rel = p.resolve().relative_to(self.data_dir.resolve())
        except Exception:
            rel = Path(p.name)

        return str(rel).replace("\\", "/")


    def __init__(self,search_client,data_dir,qrels_csv_path,qrels_txt_path,trec_eval_bin,):
        self.search = search_client
        self.data_dir = Path(data_dir)
        self.qrels_csv_path = Path(qrels_csv_path)
        self.qrels_txt_path = Path(qrels_txt_path) if qrels_txt_path else self.data_dir / "qrels.txt"

        # Ensure that exists qrels.txt in TREC format
        self._ensure_trec_qrels()

        # Find the trec_eval.exe
        if trec_eval_bin is None:
            self.trec_eval_bin = self._find_trec_eval()
        else:
            self.trec_eval_bin = Path(trec_eval_bin)

        if not self.trec_eval_bin or not self.trec_eval_bin.exists():
            raise SystemExit(
                "trec_eval binary not found. Put trec_eval.exe στον φάκελο trec_eval "
                "ή δώσε σωστό path στον constructor."
            )

        if not self.qrels_txt_path.exists():
            raise SystemExit(f"qrels file not found: {self.qrels_txt_path}")

        print(f"Using trec_eval at: {self._rel(self.trec_eval_bin)}")
        print(f"Using qrels file:  {self._rel(self.qrels_txt_path)}")

    # Reads qrels.csv and writes qrels.txt in teh form: qid 0 docid rel.
    def _fix_and_write_qrels(self):
        if not self.qrels_csv_path.exists():
            raise SystemExit(f"qrels.csv not found: {self.qrels_csv_path}")

        dfq = pd.read_csv(
            self.qrels_csv_path,
            sep=None,
            engine="python",
            encoding="utf-8-sig",
        )
        dfq = dfq.dropna(how="all")
        dfq = dfq.astype(str).applymap(lambda x: x.strip())
        cols = list(dfq.columns)

        # optional: docids from one sample results file
        res_docids = set()
        sample_results_path = self.data_dir / "results_20.txt"
        if sample_results_path.exists():
            with open(sample_results_path, encoding="utf-8", errors="replace") as f:
                for ln in f:
                    parts = re.split(r"\s+", ln.strip())
                    if len(parts) >= 3:
                        res_docids.add(parts[2])

        candidates = []
        for c in cols:
            vals = dfq[c].dropna().astype(str).str.strip().unique()[:200].tolist()
            inter = len(set(vals) & res_docids) if res_docids else 0
            num_like = sum(1 for v in vals if re.match(r"^\d+$", v))
            candidates.append((c, inter, num_like, vals[:5]))

        docid_col = max(candidates, key=lambda x: (x[1], x[2]))[0]
        qid_col = cols[0]
        rel_col = None
        for c, inter, num_like, vals in candidates:
            if c in (qid_col, docid_col):
                continue
            sample_vals = dfq[c].dropna().astype(str).str.strip().unique()[:50].tolist()
            if sample_vals and all(re.match(r"^\d+$", v) for v in sample_vals):
                rel_col = c
                break
        if rel_col is None:
            others = [c for c in cols if c not in (qid_col, docid_col)]
            rel_col = others[-1] if others else cols[-1]

        with open(self.qrels_txt_path, "w", encoding="utf-8") as out:
            for _, row in dfq.iterrows():
                qid = str(row[qid_col]).strip()
                docid = str(row[docid_col]).strip()
                rel = str(row[rel_col]).strip()
                if not qid or not docid or docid.upper() == "Q0":
                    continue
                out.write(f"{qid} 0 {docid} {rel}\n")

        print(
            f"Wrote TREC qrels -> {self._rel(self.qrels_txt_path)} "
            f"(qid_col={qid_col}, docid_col={docid_col}, rel_col={rel_col})"
        )

    def _ensure_trec_qrels(self):
        """Uses qrels.txt if exists, otherwise creates it from qrels.csv"""
        if self.qrels_txt_path.exists():
            print(f"Using existing TREC qrels: {self._rel(self.qrels_txt_path)}")
            return
        print("TREC qrels not found, creating from qrels.csv ...")
        self._fix_and_write_qrels()

    # Run trec_eval.exe
    def _find_trec_eval(self):
        candidates = [
            self.data_dir / "trec_eval" / "trec_eval.exe",
            self.data_dir / "trec_eval" / "trec_eval",
            self.data_dir / "trec_eval.exe",
            self.data_dir / "trec_eval",
        ]
        for p in candidates:
            if p.exists():
                return p

        p_on_path = shutil.which("trec_eval") or shutil.which("trec_eval.exe")
        if p_on_path:
            return Path(p_on_path)

        if self.data_dir.exists():
            for p in self.data_dir.rglob("trec_eval*"):
                if p.is_file():
                    return p
        return None

    def generate_top200_texts(self, queries_path: Path):
        df_queries = pd.read_csv(queries_path, encoding="utf-8")
        print(f"Loaded queries: {len(df_queries)}")

        TOP_K = 200
        texts_by_query = {}

        for _, row in df_queries.iterrows():
            qid = str(row.iloc[0]).strip()
            qtext = str(row.iloc[1])

            results = self.search.search_query(qtext, TOP_K)[:TOP_K]

            # keep REAL doc id + text
            docs = [
                (hit["_source"]["id"], hit["_source"]["text"])
                for hit in results
            ]

            texts_by_query[qid] = docs

        print("Retrieved top-200 docs (id + text) for all queries.")
        return texts_by_query


    def _run_trec_eval_for_file(self, results_file: Path):
        if not results_file.exists():
            raise SystemExit(f"results file not found: {results_file}")

        cmd = [
            str(self.trec_eval_bin),
            str(self.qrels_txt_path),
            str(results_file),
            "-m", "all_trec",
        ]

        print("Running:", " ".join(cmd))

        proc = subprocess.run(cmd, capture_output=True, text=True)

        if proc.returncode != 0:
            print("--- trec_eval stderr ---")
            print(proc.stderr)
            raise SystemExit(f"trec_eval failed for {results_file} (rc={proc.returncode})")

        parsed = {}
        for line in proc.stdout.strip().splitlines():
            parts = re.split(r"\s+", line.strip())
            if len(parts) < 3:
                continue

            metric, target, value_str = parts[0], parts[1], parts[-1]

            if target.lower() != "all":
                continue

            if metric.lower() == "map":
                parsed["MAP"] = float(value_str); continue

            m = re.match(r"^P_(\d+)$", metric)
            if m:
                parsed[f"P@{int(m.group(1))}"] = float(value_str)

        return {
            "P@5":  parsed.get("P@5"),
            "P@10": parsed.get("P@10"),
            "P@15": parsed.get("P@15"),
            "P@20": parsed.get("P@20"),
            "MAP":  parsed.get("MAP"),
        }


    # Evaluation με trec_eval
    def evaluate_with_trec_eval(self, ks, out_summary_path: Path = None):
        summary_rows = []

        for k in ks:
            results_file = self.data_dir / f"results_{k}.txt"
            metrics = self._run_trec_eval_for_file(results_file)
            row = {"retrieval_k": k}
            row.update(metrics)
            summary_rows.append(row)

        df_trec = (
            pd.DataFrame(summary_rows)
            .sort_values("retrieval_k")
            .reset_index(drop=True)
        )

        # Clean, official-looking table
        styled = (
            df_trec.style
            .format("{:.4f}", subset=["P@5", "P@10", "P@15", "P@20", "MAP"])
            .set_properties(**{
                "text-align": "center",
                "font-size": "13px",
            })
        )

        print("TREC Evaluation Summary:")
        display(styled)

        # Optional: write summary file
        if out_summary_path is not None:
            with open(out_summary_path, "w", encoding="utf-8") as f:
                for _, r in df_trec.iterrows():
                    k = int(r["retrieval_k"])
                    f.write(f"# summary for results_{k}.txt\n")
                    for kk in [5, 10, 15, 20]:
                        val = r.get(f"P@{kk}")
                        f.write(f"P_{kk}\tall\t{val:.4f}\n")
                    f.write(f"map\tall\t{r['MAP']:.4f}\n\n")

        return df_trec

In [160]:
#Initiate search class and index name
search=Search()
INDEX_NAME = "ir2025" 

Connected to Elasticsearch!
{'cluster_name': 'elasticsearch',
 'cluster_uuid': 'LhE-1-1hSHGWtwOEbrNFSA',
 'name': 'PPAVLOU',
 'tagline': 'You Know, for Search',
 'version': {'build_date': '2025-09-16T22:05:19.073893347Z',
             'build_flavor': 'default',
             'build_hash': '0b7fe68d2e369469ff9e9f344ab6df64ab9c5293',
             'build_snapshot': False,
             'build_type': 'zip',
             'lucene_version': '10.2.2',
             'minimum_index_compatibility_version': '8.0.0',
             'minimum_wire_compatibility_version': '8.19.0',
             'number': '9.1.4'}}


In [161]:
#Delete index if exists
if search.exists(INDEX_NAME):
    search.delete_index(index_name=INDEX_NAME)
    
#Create new empty index
search.create_index()

#Index docs
search.index_documents(DOCS_JSONL,INDEX_NAME)

Index 'ir2025' deleted successfully.
Index created with English analyzer and BM25 similarity.
Successfully indexed 18316 documents into 'ir2025'


In [168]:
evaluator = Evaluation_BM25(
    search_client=search,
    data_dir=IR_DIR,
    qrels_csv_path=IR_DIR / "qrels.csv",  
    qrels_txt_path=IR_DIR / "qrels.txt",
    trec_eval_bin=IR_DIR / "trec_eval" / "trec_eval.exe"
)

texts_by_query = evaluator.generate_top200_texts(QUERIES_CSV)


Using existing TREC qrels: qrels.txt
Using trec_eval at: trec_eval/trec_eval.exe
Using qrels file:  qrels.txt
Loaded queries: 10
Retrieved top-200 docs (id + text) for all queries.


In [169]:
print(len(texts_by_query["Q01"]))   # should be 200
print(texts_by_query["Q01"][0])     # first retrieved text


200
('193378', 'Optimodal European Travel Ecosystem: EuTravel aims to: 1. Support the EU agenda towards an open and single market for mobility services by enabling travellers to organise a multimodal trip in accordance with their own criteria including environmental performance, providing multimodal travel service providers an effective way to deliver customised services addressing any type of specialised travel needs and facilitating fact-based EU policy making. 2. Promote the creation of content, open and linked data for travellers enriching the travelling experience. 3. Support travel industry players join forces towards realising an EU shared seamless mobility strategy and architecture. EuTravel will research and demonstrate Inter-modal travel optimised with respect to synchronisation between modes, passenger experience and rights and environmental performance (Optimodal Travel). The project objectives will be realised by: 1. Developing an open and readily usable Optimodality Frame

In [170]:
def load_or_create_embeddings(texts_by_query, query_df):
    EMBED_DIR.mkdir(parents=True, exist_ok=True)

    if DOC_EMBED_FILE.exists() and QUERY_EMBED_FILE.exists():
        print("📦 Found cached embeddings — loading from disk...")
        doc_emb = np.load(DOC_EMBED_FILE)
        query_emb = np.load(QUERY_EMBED_FILE)
        print(f"✔ Loaded doc embeddings:   {doc_emb.shape}")
        print(f"✔ Loaded query embeddings: {query_emb.shape}")
        return doc_emb, query_emb

    print("⚠ No cached embeddings found — generating new ones")

    # --- Collect documents (top-200 per query) ---
    retrieved_texts = []
    for docs in texts_by_query.values():
        retrieved_texts.extend([t for _, t in docs])

    print(f"Docs to embed (retrieved only): {len(retrieved_texts)}")

    # --- Collect the actual QUERY TEXTS ---
    query_texts = query_df["Text"].astype(str).tolist()
    print(f"Queries to embed: {len(query_texts)}")

    # ---------- DOC EMBEDDINGS ----------
    print("⚙ Generating NEW document embeddings...")
    doc_emb = model.encode(
        retrieved_texts,
        batch_size=DOC_BATCH,
        convert_to_numpy=True,
        show_progress_bar=True
    )
    np.save(DOC_EMBED_FILE, doc_emb)
    print(f"✔ Saved doc embeddings → {DOC_EMBED_FILE.resolve()}")

    # ---------- QUERY EMBEDDINGS ----------
    print("⚙ Generating NEW query embeddings...")
    query_emb = model.encode(
        query_texts,
        batch_size=QUERY_BATCH,
        convert_to_numpy=True,
        show_progress_bar=True
    )
    np.save(QUERY_EMBED_FILE, query_emb)
    print(f"✔ Saved query embeddings → {QUERY_EMBED_FILE.resolve()}")

    return doc_emb, query_emb


doc_embeddings, query_embeddings = load_or_create_embeddings(
    texts_by_query,
    df_queries
)


📦 Found cached embeddings — loading from disk...
✔ Loaded doc embeddings:   (2000, 768)
✔ Loaded query embeddings: (10, 768)


In [171]:
def build_faiss_index(embeddings, save_path=None, nlist=100, nprobe=10):
    emb = embeddings.astype("float32")
    dim = emb.shape[1]

    # Cosine similarity = normalize + inner-product
    faiss.normalize_L2(emb)
    quantizer = faiss.IndexFlatIP(dim)

    index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)

    index.train(emb)
    index.add(emb)
    index.nprobe = nprobe

    if save_path:
        faiss.write_index(index, str(save_path))

    return index


In [172]:
def expand_query_with_prf(q_emb, index,
                          top_m=PRF_TOP_M,
                          alpha=PRF_ALPHA,
                          beta=PRF_BETA,
                          candidate_pool=REQUERY_CANDIDATES):

    # normalize query (cosine search)
    q = q_emb.astype("float32")
    faiss.normalize_L2(q.reshape(1, -1))

    # retrieve candidate documents
    _, I = index.search(q.reshape(1, -1), candidate_pool)
    top_idxs = I[0][:top_m]
    if len(top_idxs) == 0:
        return q_emb

    # mean embedding of feedback docs
    top_embs = doc_embeddings[top_idxs].astype("float32")
    faiss.normalize_L2(top_embs)
    feedback_mean = top_embs.mean(axis=0)

    # blend query + feedback
    expanded = alpha * q_emb + beta * feedback_mean

    # renormalize
    expanded /= (np.linalg.norm(expanded) + 1e-12)
    return expanded


In [173]:
def generate_faiss_hybrid_results(
    df_queries,
    texts_by_query,
    query_embeddings,
    ks=(20, 30, 50),
    prf_enabled=PRF_ENABLED
):

    out_files = {
        k: (IR_DIR / f"results_hybrid_{k}.txt").open("w", encoding="utf-8")
        for k in ks
    }

    print("🚀 Running hybrid ES → FAISS reranking...")
    print(f"Queries={len(df_queries)}, ks={ks}, PRF={prf_enabled}")
    print("-" * 60)

    for q_idx, row in df_queries.iterrows():
        qid = str(row["ID"])
        q_emb = query_embeddings[q_idx].astype("float32")

        docs = texts_by_query[qid]          # (docid, text)
        doc_ids  = [d for d, _ in docs]
        doc_texts = [t for _, t in docs]

        # ---- embed only docs of this query ----
        doc_embs = model.encode(doc_texts, convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(doc_embs)
        dim = doc_embs.shape[1]

        index = faiss.IndexFlatIP(dim)
        index.add(doc_embs)

        # ---- query vector ----
        if prf_enabled:
            expanded = expand_query_with_prf(q_emb, index)
            q_vec = expanded.reshape(1, -1)
            faiss.normalize_L2(q_vec)
            prf_flag = "PRF"
        else:
            q_vec = q_emb.reshape(1, -1)
            prf_flag = "NO-PRF"

        # ---- run search ----
        D, I = index.search(q_vec, max(ks))

        # ---- write outputs ----
        for k in ks:
            for rank, (doc_idx, score) in enumerate(zip(I[0][:k], D[0][:k]), start=1):
                docid = doc_ids[doc_idx]
                out_files[k].write(
                    f"{qid} Q0 {docid} {rank} {float(score):.6f} HYBRID\n"
                )

        # ---- single debug line per query ----
        print(
            f"[Q{qid}] docs={len(docs)}  dim={dim}  "
            f"top={max(ks)}  mode={prf_flag}  ✓ done"
        )

    for f in out_files.values():
        f.close()

    print("-" * 60)
    print("✔ Created:", ", ".join([f"results_hybrid_{k}.txt" for k in ks]))


In [ ]:
generate_faiss_hybrid_results(
    df_queries,
    texts_by_query,
    query_embeddings,
    ks=(20, 30, 50),
    prf_enabled=PRF_ENABLED
)


🚀 Running hybrid ES → FAISS reranking...
Queries=10, ks=(20, 30, 50), PRF=False
------------------------------------------------------------
[QQ01] docs=200  dim=768  top=50  mode=NO-PRF  ✓ done
[QQ02] docs=200  dim=768  top=50  mode=NO-PRF  ✓ done
[QQ03] docs=200  dim=768  top=50  mode=NO-PRF  ✓ done


In [ ]:
class Evaluation_FAISS:
    # ---------- path helper ----------
    def _rel(self, p: Path):
        try:
            rel = p.resolve().relative_to(self.data_dir.resolve())
        except Exception:
            rel = Path(p.name)
        return str(rel).replace("\\", "/")

    # ---------- init ----------
    def __init__(self, search_client, data_dir, qrels_csv_path, qrels_txt_path, trec_eval_bin):
        self.search = search_client
        self.data_dir = Path(data_dir)

        self.qrels_csv_path = Path(qrels_csv_path)
        self.qrels_txt_path = Path(qrels_txt_path) if qrels_txt_path else self.data_dir / "qrels.txt"

        # ensure qrels.txt exists
        self._ensure_trec_qrels()

        # locate trec_eval
        self.trec_eval_bin = Path(trec_eval_bin) if trec_eval_bin else self._find_trec_eval()

        if not self.trec_eval_bin or not self.trec_eval_bin.exists():
            raise SystemExit(
                "trec_eval binary not found. Put trec_eval.exe in trec_eval folder "
                "or pass correct path to constructor."
            )

        if not self.qrels_txt_path.exists():
            raise SystemExit(f"qrels file not found: {self.qrels_txt_path}")

        print(f"Using trec_eval at: {self._rel(self.trec_eval_bin)}")
        print(f"Using qrels file:  {self._rel(self.qrels_txt_path)}")

    # ---------- build qrels.txt ----------
    def _fix_and_write_qrels(self):
        if not self.qrels_csv_path.exists():
            raise SystemExit(f"qrels.csv not found: {self.qrels_csv_path}")

        dfq = pd.read_csv(self.qrels_csv_path, sep=None, engine="python", encoding="utf-8-sig")
        dfq = dfq.dropna(how="all")
        dfq = dfq.astype(str).applymap(lambda x: x.strip())
        cols = list(dfq.columns)

        # optional docids from FAISS top-200 results
        res_docids = set()
        sample_results_path = self.data_dir / "results_faiss_200.txt"
        if sample_results_path.exists():
            with open(sample_results_path, encoding="utf-8", errors="replace") as f:
                for ln in f:
                    parts = re.split(r"\s+", ln.strip())
                    if len(parts) >= 3:
                        res_docids.add(parts[2])

        candidates = []
        for c in cols:
            vals = dfq[c].dropna().astype(str).str.strip().unique()[:200].tolist()
            inter = len(set(vals) & res_docids) if res_docids else 0
            num_like = sum(1 for v in vals if re.match(r"^\d+$", v))
            candidates.append((c, inter, num_like, vals[:5]))

        docid_col = max(candidates, key=lambda x: (x[1], x[2]))[0]
        qid_col = cols[0]

        # pick relevance col
        rel_col = None
        for c, inter, num_like, vals in candidates:
            if c in (qid_col, docid_col):
                continue
            sample_vals = dfq[c].dropna().astype(str).str.strip().unique()[:50].tolist()
            if sample_vals and all(re.match(r"^\d+$", v) for v in sample_vals):
                rel_col = c
                break
        if rel_col is None:
            others = [c for c in cols if c not in (qid_col, docid_col)]
            rel_col = others[-1] if others else cols[-1]

        with open(self.qrels_txt_path, "w", encoding="utf-8") as out:
            for _, row in dfq.iterrows():
                qid = str(row[qid_col]).strip()
                docid = str(row[docid_col]).strip()
                rel = str(row[rel_col]).strip()
                if not qid or not docid or docid.upper() == "Q0":
                    continue
                out.write(f"{qid} 0 {docid} {rel}\n")

        print(
            f"Wrote TREC qrels -> {self._rel(self.qrels_txt_path)} "
            f"(qid_col={qid_col}, docid_col={docid_col}, rel_col={rel_col})"
        )

    def _ensure_trec_qrels(self):
        if self.qrels_txt_path.exists():
            print(f"Using existing TREC qrels: {self._rel(self.qrels_txt_path)}")
            return
        print("TREC qrels not found, creating from qrels.csv ...")
        self._fix_and_write_qrels()

    # ---------- locate trec_eval ----------
    def _find_trec_eval(self):
        candidates = [
            self.data_dir / "trec_eval" / "trec_eval.exe",
            self.data_dir / "trec_eval" / "trec_eval",
            self.data_dir / "trec_eval.exe",
            self.data_dir / "trec_eval",
        ]
        for p in candidates:
            if p.exists():
                return p

        p_on_path = shutil.which("trec_eval") or shutil.which("trec_eval.exe")
        if p_on_path:
            return Path(p_on_path)

        if self.data_dir.exists():
            for p in self.data_dir.rglob("trec_eval*"):
                if p.is_file():
                    return p
        return None

    # ---------- run trec_eval ----------
    def _run_trec_eval_for_file(self, results_file: Path):
        if not results_file.exists():
            raise SystemExit(f"results file not found: {results_file}")

        cmd = [
            str(self.trec_eval_bin),
            str(self.qrels_txt_path),
            str(results_file),
            "-m", "all_trec",
        ]

        print("Running:", " ".join(cmd))
        proc = subprocess.run(cmd, capture_output=True, text=True)

        if proc.returncode != 0:
            print("--- trec_eval stderr ---")
            print(proc.stderr)
            raise SystemExit(f"trec_eval failed for {results_file} (rc={proc.returncode})")

        parsed = {}
        for line in proc.stdout.strip().splitlines():
            parts = re.split(r"\s+", line.strip())
            if len(parts) < 3:
                continue

            metric, target, value_str = parts[0], parts[1], parts[-1]
            if target.lower() != "all":
                continue

            if metric.lower() == "map":
                parsed["MAP"] = float(value_str)
                continue

            m = re.match(r"^P_(\d+)$", metric)
            if m:
                parsed[f"P@{int(m.group(1))}"] = float(value_str)

        return {
            "P@5":  parsed.get("P@5"),
            "P@10": parsed.get("P@10"),
            "P@15": parsed.get("P@15"),
            "P@20": parsed.get("P@20"),
            "MAP":  parsed.get("MAP"),
        }

    # ---------- evaluate FAISS Top-200 ----------
    def evaluate_with_trec_eval(self, ks=(200,), out_summary_path: Path = None):
        summary_rows = []

        for k in ks:
            results_file = self.data_dir / f"results_hybrid_{k}.txt"
            metrics = self._run_trec_eval_for_file(results_file)

            row = {"retrieval_k": k}
            row.update(metrics)
            summary_rows.append(row)

        df_trec = (
            pd.DataFrame(summary_rows)
            .sort_values("retrieval_k")
            .reset_index(drop=True)
        )

        print("TREC Evaluation Summary:")
        print(df_trec)

        if out_summary_path is not None:
            with open(out_summary_path, "w", encoding="utf-8") as f:
                for _, r in df_trec.iterrows():
                    k = int(r["retrieval_k"])
                    f.write(f"# summary for results_faiss_{k}.txt\n")
                    for kk in [5, 10, 15, 20]:
                        val = r.get(f"P@{kk}")
                        f.write(f"P_{kk}\tall\t{val:.4f}\n")
                    f.write(f"map\tall\t{r['MAP']:.4f}\n\n")

        return df_trec


In [ ]:
evaluator = Evaluation_FAISS(
    search_client=None,
    data_dir=IR_DIR,
    qrels_csv_path=QRELS_CSV,
    qrels_txt_path=QRELS_TXT,
    trec_eval_bin=TREC_EVAL_BIN
)

RESULT_KS = (20, 30, 50)

df_hybrid = evaluator.evaluate_with_trec_eval(
    ks=RESULT_KS,
    out_summary_path=IR_DIR / "trec_eval_hybrid_summary3.txt"
)



Using existing TREC qrels: qrels.txt
Using trec_eval at: trec_eval/trec_eval.exe
Using qrels file:  qrels.txt
Running: c:\Users\perik\Downloads\Semantic-Information-Retrieval-System-ElasticSearch-FAISS-Transformers-\IR2025\trec_eval\trec_eval.exe c:\Users\perik\Downloads\Semantic-Information-Retrieval-System-ElasticSearch-FAISS-Transformers-\IR2025\qrels.txt c:\Users\perik\Downloads\Semantic-Information-Retrieval-System-ElasticSearch-FAISS-Transformers-\IR2025\results_hybrid_20.txt -m all_trec
Running: c:\Users\perik\Downloads\Semantic-Information-Retrieval-System-ElasticSearch-FAISS-Transformers-\IR2025\trec_eval\trec_eval.exe c:\Users\perik\Downloads\Semantic-Information-Retrieval-System-ElasticSearch-FAISS-Transformers-\IR2025\qrels.txt c:\Users\perik\Downloads\Semantic-Information-Retrieval-System-ElasticSearch-FAISS-Transformers-\IR2025\results_hybrid_30.txt -m all_trec
Running: c:\Users\perik\Downloads\Semantic-Information-Retrieval-System-ElasticSearch-FAISS-Transformers-\IR2025

In [ ]:
print(IR_DIR)

c:\Users\perik\Downloads\Semantic-Information-Retrieval-System-ElasticSearch-FAISS-Transformers-\IR2025
